# NHL intro — sportsdataverse-py

Two hockey surfaces live under `sportsdataverse.nhl`:

1. **NHL api-web (native, `nhl_*`)** — the league's own modern feed: schedule, play-by-play (`nhl_web_pbp`), boxscores, standings, rosters, player game logs, plus player tracking (`nhl_edge_*`) and the stats-rest / records APIs (`nhl_stats_rest_*`, `nhl_records_*`). These return tidy frames directly (`return_as_pandas=True`).
2. **ESPN NHL (`espn_nhl_*`)** — the same ESPN conventions used across every other league: `espn_nhl_teams`, `espn_nhl_schedule`, `espn_nhl_pbp` (a dict), `espn_nhl_standings`, etc.

Plus the **`load_nhl_*` parquet loaders** that read pre-built data releases.

R companion: [fastRhockey](https://fastRhockey.sportsdataverse.org) (NHL + PWHL). Python neighbor: [nhl-api-py](https://github.com/coreyjs/nhl-api-py). Part of the [SportsDataverse](https://py.sportsdataverse.org/docs/ecosystem).

## Setup

```sh
pip install sportsdataverse
```

In [ ]:
import polars as pl
import sportsdataverse as sdv

## NHL api-web (native)

The native wrappers hit the league's `api-web.nhle.com` feed. Every one accepts `return_as_pandas=True` to get a tidy frame back (the default `return_parsed=True` already shapes the JSON into rows). Native game IDs look like `2023030417` (season + game-type + sequence), which is **different** from ESPN's `401675111` for the same game.

We'll use the 2024 Stanley Cup Final Game 7 throughout: Florida Panthers 2, Edmonton Oilers 1 (June 24, 2024).

### Native: schedule

`nhl_web_schedule(date='YYYY-MM-DD')` returns the day's games with `home_team_*` / `away_team_*` columns and the native `id`.

In [ ]:
nat_sched = sdv.nhl.nhl_web_schedule(date='2024-06-24', return_as_pandas=True)
pl.from_pandas(nat_sched).select([
    'id', 'game_state',
    'home_team_abbrev', 'home_team_score',
    'away_team_abbrev', 'away_team_score',
]).head()

### Native: play-by-play

`nhl_web_pbp(game_id=...)` returns one row per event. Columns use `snake_case` (`type_desc_key`, `time_in_period`, `period_descriptor_number`) — not ESPN dot-notation.

In [ ]:
nat_pbp = pl.from_pandas(sdv.nhl.nhl_web_pbp(game_id=2023030417, return_as_pandas=True))
print(nat_pbp.shape)
nat_pbp.select([
    'period_descriptor_number', 'time_in_period', 'type_desc_key',
    'details_event_owner_team_id', 'details_x_coord', 'details_y_coord',
]).head()

In [ ]:
# Event-type mix for the game (native uses `type_desc_key`, e.g. shot-on-goal, goal, hit)
(nat_pbp
    .group_by('type_desc_key')
    .agg(pl.len().alias('events'))
    .sort('events', descending=True)
    .head(10))

### Native: boxscore

`nhl_boxscore(game_id=...)` returns one row per player (skaters + goalies) with `home_away`, `position`, and per-player stats.

In [ ]:
box = pl.from_pandas(sdv.nhl.nhl_boxscore(game_id=2023030417, return_as_pandas=True))
(box
    .filter(pl.col('position') != 'G')
    .select(['name_default', 'home_away', 'position', 'goals', 'assists', 'points', 'sog', 'toi'])
    .sort('points', descending=True)
    .head())

### Native: standings

`nhl_standings(date='YYYY-MM-DD')` returns one row per team with conference/division context and points.

In [ ]:
standings = pl.from_pandas(sdv.nhl.nhl_standings(date='2024-04-15', return_as_pandas=True))
(standings
    .select(['team_name_default', 'conference_name', 'division_name', 'games_played', 'wins', 'losses', 'points'])
    .sort('points', descending=True)
    .head())

### Native: roster & player game log

`nhl_roster(team=ABBREV, season=...)` lists a club's roster; `nhl_player_game_log(player_id=..., season=...)` returns a player's game-by-game line. Connor McDavid is `8478402`; the season string is `20232024`.

In [ ]:
roster = pl.from_pandas(sdv.nhl.nhl_roster(team='FLA', season=20232024, return_as_pandas=True))
roster.select(['id', 'first_name_default', 'last_name_default', 'sweater_number', 'position_code', 'shoots_catches']).head()

In [ ]:
gamelog = pl.from_pandas(sdv.nhl.nhl_player_game_log(player_id=8478402, season=20232024, return_as_pandas=True))
print(gamelog.shape)
gamelog.select(['game_date', 'opponent_abbrev', 'goals', 'assists', 'points', 'shots', 'toi']).head()

### Native: NHL EDGE (player tracking)

`nhl_edge_*` wraps the NHL EDGE puck-and-player tracking surface. The `*_landing` calls return a wide single-row leaderboard frame; per-player `*_detail` calls return a player's tracked values alongside the league average and percentile. Here's McDavid's skating-speed detail for 2023-24.

In [ ]:
edge = pl.from_pandas(
    sdv.nhl.nhl_edge_skater_skating_speed_detail(player_id=8478402, season=20232024, return_as_pandas=True)
)
edge.select([
    'skating_speed_details_max_skating_speed_imperial',
    'skating_speed_details_max_skating_speed_league_avg_imperial',
    'skating_speed_details_bursts_over22_value',
    'skating_speed_details_bursts_over22_percentile',
])

### Native: stats-rest leaders

`nhl_stats_rest_leaders_skaters(attribute=...)` taps the `api.nhle.com/stats/rest` leaderboards — a clean top-10 frame per attribute (e.g. `goals`, `points`, `assists`).

In [ ]:
leaders = pl.from_pandas(
    sdv.nhl.nhl_stats_rest_leaders_skaters(attribute='goals', return_as_pandas=True)
)
leaders.select(['player_full_name', 'player_position_code', 'team_tri_code', 'goals']).head(10)

## ESPN NHL (`espn_nhl_*`)

ESPN's surface follows the same conventions as every other league in the package: team-name columns are `home_display_name` / `away_display_name`, scores come back as **strings**, and `espn_nhl_pbp` returns a **dict** whose `plays` use raw ESPN dot-notation. ESPN game IDs look like `401675111`.

### ESPN: teams

In [ ]:
teams = sdv.nhl.espn_nhl_teams()
print(teams.shape)
teams.select(['team_id', 'team_location', 'team_name', 'team_abbreviation', 'team_display_name']).head()

### ESPN: schedule

`espn_nhl_schedule(dates=YYYYMMDD)`. Team names are `home_display_name` / `away_display_name`; scores are strings, so cast before doing arithmetic.

In [ ]:
schedule = sdv.nhl.espn_nhl_schedule(dates=20240624)
schedule.select([
    'id', 'home_display_name', 'away_display_name',
    pl.col('home_score').cast(pl.Int64, strict=False).alias('home_score'),
    pl.col('away_score').cast(pl.Int64, strict=False).alias('away_score'),
]).head()

### ESPN: play-by-play

`espn_nhl_pbp(game_id=...)` returns a **dict** (keys like `plays`, `boxscore`, `header`, ...). `pbp['plays']` is a **list of raw dicts** — build a frame with `pl.DataFrame(..., infer_schema_length=None)`. Columns use ESPN dot-notation: `period.number`, `clock.displayValue`, `scoringPlay`, `shootingPlay`, `type.text`, `coordinate.x` / `coordinate.y`.

In [ ]:
pbp = sdv.nhl.espn_nhl_pbp(game_id=401675111)
list(pbp.keys())[:8]

In [ ]:
plays = pl.DataFrame(pbp['plays'], infer_schema_length=None)
print(plays.shape)
plays.select(['period.number', 'clock.displayValue', 'text', 'type.text', 'scoringPlay']).head()

### ESPN: shot-event filter

Filter the ESPN play-by-play to shooting plays — the simplest analysis primitive in hockey. `shootingPlay` is a Boolean column.

In [ ]:
shots = plays.filter(pl.col('shootingPlay') == True)
shots.shape

In [ ]:
(shots
    .group_by('type.text')
    .agg(pl.len().alias('events'))
    .sort('events', descending=True)
    .head(10))

## Parquet loaders (`load_nhl_*`)

The `load_nhl_*` loaders read pre-built parquet data releases (fastRhockey-era schema) and return polars frames — fast for multi-season work and offline-friendly. Pass `seasons=[...]`; add `return_as_pandas=True` for a pandas frame.

In [ ]:
schedule_2024 = sdv.nhl.load_nhl_schedule(seasons=[2024])
print(schedule_2024.shape)
schedule_2024.select(['game_id', 'game_date', 'home_team_name', 'away_team_name', 'home_score', 'away_score']).head()

In [ ]:
team_box = sdv.nhl.load_nhl_team_box(seasons=[2024])
print(team_box.shape)
team_box.select(['game_id', 'team_name', 'goals', 'shots', 'hits', 'power_play_goals']).head()

## Pipeline example: goals per period

Combine the ESPN play-by-play with polars: filter to scoring plays and group by period. (Game 7 ended 2-1, so this is a small, easy-to-read frame.)

In [ ]:
(plays
    .filter(pl.col('scoringPlay') == True)
    .group_by('period.number')
    .agg(pl.len().alias('goals'))
    .sort('period.number'))

## Cross-references

- R companion: [fastRhockey](https://fastRhockey.sportsdataverse.org) (NHL + PWHL)
- Data sources: NHL api-web (`api-web.nhle.com`), NHL stats-rest (`api.nhle.com`), NHL EDGE, and the ESPN NHL API
- Python alternative: [nhl-api-py](https://github.com/coreyjs/nhl-api-py)
- Plotting: matplotlib, plotnine

## Where to go next

- API docs: [`docs/docs/nhl/index.md`](../../docs/docs/nhl/index.md)
- Women's hockey: [`10_pwhl_intro.ipynb`](10_pwhl_intro.ipynb) (PWHL, also via fastRhockey)
- Cross-sport overview: `01_quickstart.ipynb`